In [16]:
import uuid
from typing import TypedDict, Literal, Annotated
from dotenv import load_dotenv

from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import add_messages

load_dotenv()

True

In [17]:
llm = init_chat_model(
    model="gpt-4o-mini",
    temperature=0.0,
)

In [18]:
class IntentClassifier(BaseModel):
    message_intent: Literal["chat", "knowledge", "code"] = Field(
        ...,
        description="Classify whether the user wants to just chat, ask for knowledge, or change code in the project."
    )

In [19]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    message_intent: str | None

In [20]:
def classify_intent(state: State):
    structured_llm = llm.with_structured_output(IntentClassifier)
    intent = structured_llm.invoke([
        {
            "role": "system",
            "content": "You are an intent classifier. Classify the whether the user wants to chat (chat), retrieve knowledge (knowledge), or change code (code)."},
        {
            "role": "user",
            "content": state["messages"][-1]["content"]
        }
    ])
    
    return {"message_intent": intent.message_intent}

In [21]:
def prompt_llm_chat(state: State):
    response = llm.invoke([
        {
            "role": "system",
            "content": "You are a helpful assistant. Answer the user's questions to the best of your ability."
        },
        *state["messages"]
    ])
    
    return {"messages": state["messages"] + [response]}

In [22]:
def prompt_llm_rag(state: State):
    response = llm.invoke([
        {
            "role": "system",
            "content": "No matter what the user says, always say I'm the RAG agent"
        },
        *state["messages"]
    ])
    
    return {"messages": state["messages"] + [response]}

In [23]:
def prompt_llm_code(state: State):
    response = llm.invoke([
        {
            "role": "system",
            "content": "No matter what the user says, always say I'm the code agent"
        },
        *state["messages"]
    ])
    
    return {"messages": state["messages"] + [response]}

In [24]:
graph_builder = StateGraph(State)

In [25]:
graph_builder.add_node('classifier', classify_intent)
graph_builder.add_node('chat_agent', prompt_llm_chat)
graph_builder.add_node('rag_agent', prompt_llm_rag)
graph_builder.add_node('code_agent', prompt_llm_code)

In [26]:
graph_builder.add_edge(START, 'classifier')
graph_builder.add_conditional_edges(
    'classifier',
    lambda state: state['message_intent'], {'chat': 'chat_agent', 'knowledge': 'rag_agent', 'code': 'code_agent'}
)

In [27]:
graph_builder.add_edge('chat_agent', END)
graph_builder.add_edge('rag_agent', END)
graph_builder.add_edge('code_agent', END)

In [28]:
checkpoint_saver = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpoint_saver)

config = {
    'configurable': {'thread_id': uuid.uuid4()}
}   

In [29]:
while True:
    user_message = input('Enter message:')
    result = graph.invoke({'messages': [{'role': 'user', 'content': user_message}]}, config=config)
    
    print(result.messages[-1]["content"])

TypeError: 'HumanMessage' object is not subscriptable